<a href="https://colab.research.google.com/github/jaredlan1/getting_started_with_ML/blob/main/ML_Final_Project_CNN_KAN_Colab_v6_ppt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN + KAN FTIR Pipeline with PowerPoint Artifacts

This notebook/script version adds the missing presentation outputs:
- FTIR spectrum overview plot
- performance comparison tables for Slide 8
- predicted-vs-actual plots for Slide 9
- symbolic equation summaries for Slide 10
- training curves and CNN+MLP vs CNN+KAN ablation for Slide 11

It writes PowerPoint-ready PNG/CSV/TXT files to `presentation_artifacts/` after training.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
from IPython.display import display

import copy
import math
import os
import re
import textwrap
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    from efficient_kan import KAN as EfficientKAN
except ImportError:
    import subprocess, sys
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'git+https://github.com/Blealtan/efficient-kan.git',
    ])
    from efficient_kan import KAN as EfficientKAN

try:
    from kan import KAN as SymbolicKAN
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pykan'])
    from kan import KAN as SymbolicKAN

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


def set_global_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def sanitize_filename(name):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(name)).strip('_')

Using device: cuda


## Data loading

In [5]:
def load_ftir_data(base_directory):
    """
    Load FTIR spectra/label CSV pairs from a directory.

    Expected filenames:
    - spectra_ec_<X>_emc_<Y>.csv
    - mole_fractions_ec_<X>_emc_<Y>.csv
    """
    all_spectra_data = []
    all_labels_data = []
    processed_count = 0

    file_pattern = re.compile(r'(spectra|mole_fractions)_ec_(\d+)_emc_(\d+)\.csv')

    if not os.path.isdir(base_directory):
        print(f"Error: Base directory '{base_directory}' does not exist.")
        return [], []

    spectra_files_map = {}
    labels_files_map = {}

    for entry_name in os.listdir(base_directory):
        full_entry_path = os.path.join(base_directory, entry_name)
        if os.path.isfile(full_entry_path):
            file_match = file_pattern.match(entry_name)
            if file_match:
                file_type = file_match.group(1)
                ec_val = file_match.group(2)
                emc_val = file_match.group(3)
                key = (ec_val, emc_val)
                if file_type == 'spectra':
                    spectra_files_map[key] = full_entry_path
                elif file_type == 'mole_fractions':
                    labels_files_map[key] = full_entry_path

    for key, spectra_file_path in spectra_files_map.items():
        if key in labels_files_map:
            labels_file_path = labels_files_map[key]
            try:
                spectra_df = pd.read_csv(spectra_file_path, header=None)
                labels_df = pd.read_csv(labels_file_path, header=None)
                all_spectra_data.append(spectra_df)
                all_labels_data.append(labels_df)
                processed_count += 1
            except Exception as e:
                print(f"Error loading files for key {key}: {e}")
        else:
            print(f"Warning: No corresponding mole_fractions file found for spectra key {key}. Skipping.")

    print(f"Successfully processed {processed_count} sets of FTIR data.")
    return all_spectra_data, all_labels_data


def build_clean_dataset(ftir_spectra, ftir_labels):
    all_labels_list = []
    all_spectra_list = []
    wavenumbers = None

    for file_idx, (spec_df, label_df) in enumerate(zip(ftir_spectra, ftir_labels)):
        if wavenumbers is None:
            wavenumbers = spec_df.iloc[1:, 0].values.astype(float)
        else:
            current_wn = spec_df.iloc[1:, 0].values.astype(float)
            if not np.allclose(current_wn, wavenumbers):
                raise ValueError(
                    f'Wavenumber axis mismatch detected in spectra file index {file_idx}. '
                    'Interpolation/resampling is required before model fitting.'
                )

        for col_idx in range(1, spec_df.shape[1]):
            spectrum = spec_df.iloc[1:, col_idx].values.astype(float)
            all_spectra_list.append(spectrum)

        for row_idx in range(1, len(label_df)):
            label_row = label_df.iloc[row_idx, :].values.astype(float)
            all_labels_list.append(label_row)

    X = np.array(all_spectra_list, dtype=float)
    y = np.array(all_labels_list, dtype=float)
    return X, y, wavenumbers

## Augmentation and preprocessing

In [6]:
class SpectralAugmenter:
    def __init__(self, noise_level=0.01, baseline_range=0.02, intensity_range=0.05, shift_range=5):
        self.noise_level = noise_level
        self.baseline_range = baseline_range
        self.intensity_range = intensity_range
        self.shift_range = shift_range

    def add_noise(self, spectrum):
        noise = np.random.normal(0, self.noise_level, spectrum.shape)
        return spectrum + noise

    def baseline_shift(self, spectrum):
        shift = np.random.uniform(-self.baseline_range, self.baseline_range)
        tilt = np.random.uniform(-self.baseline_range / 2, self.baseline_range / 2)
        x = np.linspace(-1, 1, len(spectrum))
        return spectrum + shift + tilt * x

    def intensity_scale(self, spectrum):
        scale = np.random.uniform(1 - self.intensity_range, 1 + self.intensity_range)
        return spectrum * scale

    def spectral_shift(self, spectrum):
        shift = np.random.randint(-self.shift_range, self.shift_range + 1)
        if shift == 0:
            return spectrum
        return np.roll(spectrum, shift)

    def augment(self, spectrum, n_augmentations=1):
        augmented = []
        for _ in range(n_augmentations):
            aug_spec = spectrum.copy()
            if np.random.rand() > 0.3:
                aug_spec = self.add_noise(aug_spec)
            if np.random.rand() > 0.3:
                aug_spec = self.baseline_shift(aug_spec)
            if np.random.rand() > 0.3:
                aug_spec = self.intensity_scale(aug_spec)
            if np.random.rand() > 0.5:
                aug_spec = self.spectral_shift(aug_spec)
            augmented.append(aug_spec)
        return np.array(augmented)


def augment_dataset(X, y, augmentation_factor=5):
    augmenter = SpectralAugmenter()
    X_augmented = [X]
    y_augmented = [y]

    for i in range(len(X)):
        aug_spectra = augmenter.augment(X[i], n_augmentations=augmentation_factor - 1)
        X_augmented.append(aug_spectra)
        y_augmented.append(np.tile(y[i], (augmentation_factor - 1, 1)))

    X_aug = np.vstack(X_augmented)
    y_aug = np.vstack(y_augmented)
    indices = np.random.permutation(len(X_aug))
    return X_aug[indices], y_aug[indices]


def preprocess_spectra(X_train, X_test=None):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    if X_test is not None:
        X_test_scaled = scaler.transform(X_test)
        return X_train_scaled, X_test_scaled, scaler
    return X_train_scaled, scaler


def make_cv_splits(X, n_folds=5, seed=SEED):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return list(kf.split(X))

## Model definitions

In [7]:
class SpectraTensorDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class CNNFeatureExtractor1D(nn.Module):
    """Preserves the original convolutional backbone from the Keras model."""

    def __init__(self, dropout_rate=0.4):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=11)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout(dropout_rate)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=7)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout(dropout_rate)

        self.conv3 = nn.Conv1d(64, 128, kernel_size=5)
        self.bn3 = nn.BatchNorm1d(128)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.drop3 = nn.Dropout(dropout_rate)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        elif x.dim() == 3 and x.shape[1] != 1 and x.shape[2] == 1:
            # TODO: Legacy channel-last tensors should be fixed upstream.
            x = x.transpose(1, 2)
        elif x.dim() != 3:
            raise ValueError(f'Expected 2D or 3D tensor, got shape {tuple(x.shape)}')

        x = self.conv1(x)
        x = torch.relu(x)
        x = self.bn1(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = torch.relu(x)
        x = self.bn2(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = torch.relu(x)
        x = self.bn3(x)
        x = self.global_pool(x).squeeze(-1)
        x = self.drop3(x)
        return x


class CNNMLPRegressor(nn.Module):
    """A PyTorch recreation of the original dense regression head for ablation."""

    def __init__(self, input_length, n_outputs=4, dropout_rate=0.4):
        super().__init__()
        self.feature_extractor = CNNFeatureExtractor1D(dropout_rate=dropout_rate)
        with torch.no_grad():
            dummy = torch.zeros(1, 1, input_length)
            feature_dim = self.feature_extractor(dummy).shape[1]

        self.feature_dim = feature_dim
        self.head = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, n_outputs),
        )

    def forward(self, x):
        features = self.feature_extractor(x)
        return self.head(features)


class CNNKANRegressor(nn.Module):
    def __init__(
        self,
        input_length,
        n_outputs=4,
        dropout_rate=0.4,
        kan_hidden_dims=(64, 32),
        grid_size=5,
        spline_order=3,
    ):
        super().__init__()
        self.feature_extractor = CNNFeatureExtractor1D(dropout_rate=dropout_rate)
        with torch.no_grad():
            dummy = torch.zeros(1, 1, input_length)
            feature_dim = self.feature_extractor(dummy).shape[1]

        self.feature_dim = feature_dim

        # Why KAN here:
        # - KAN uses learnable spline functions on edges, which can expose richer, potentially more
        #   interpretable nonlinear responses to spectroscopic features than a fixed-activation MLP.
        # - Tune these first for spectral data: (1) hidden width, (2) grid size, (3) spline order,
        #   then (4) spline regularization strength.
        # - TODO: KAN regularization is not equivalent to standard dropout. Do not pretend they are the same.
        self.kan_head = EfficientKAN(
            layers_hidden=[feature_dim, *list(kan_hidden_dims), n_outputs],
            grid_size=grid_size,
            spline_order=spline_order,
        )

    def forward(self, x, update_grid=False):
        features = self.feature_extractor(x)
        try:
            return self.kan_head(features, update_grid=update_grid)
        except TypeError:
            # TODO: Some efficient-kan versions do not expose update_grid in forward().
            return self.kan_head(features)


def build_cnn_model(model_type, input_length, n_outputs=4, dropout_rate=0.4,
                    kan_hidden_dims=(64, 32), grid_size=5, spline_order=3):
    if model_type == 'mlp':
        return CNNMLPRegressor(input_length=input_length, n_outputs=n_outputs, dropout_rate=dropout_rate)
    if model_type == 'kan':
        return CNNKANRegressor(
            input_length=input_length,
            n_outputs=n_outputs,
            dropout_rate=dropout_rate,
            kan_hidden_dims=kan_hidden_dims,
            grid_size=grid_size,
            spline_order=spline_order,
        )
    raise ValueError(f'Unsupported model_type: {model_type}')


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def _conv_l2_penalty(model):
    return (
        model.feature_extractor.conv1.weight.pow(2).sum()
        + model.feature_extractor.conv2.weight.pow(2).sum()
        + model.feature_extractor.conv3.weight.pow(2).sum()
    )


def train_torch_regressor(
    model,
    train_loader,
    val_loader,
    epochs=100,
    learning_rate=1e-3,
    patience=20,
    conv_l2_lambda=1e-3,
    kan_reg_strength=0.0,
    regularize_activation=1.0,
    regularize_entropy=1.0,
):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    best_state = None
    best_val_loss = float('inf')
    best_epoch = 0
    epochs_without_improvement = 0
    history = {'train_loss': [], 'val_loss': []}

    model.to(DEVICE)

    for epoch in range(epochs):
        model.train()
        running_train_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss = loss + conv_l2_lambda * _conv_l2_penalty(model)

            if kan_reg_strength > 0.0 and hasattr(model, 'kan_head'):
                if hasattr(model.kan_head, 'regularization_loss'):
                    loss = loss + kan_reg_strength * model.kan_head.regularization_loss(
                        regularize_activation=regularize_activation,
                        regularize_entropy=regularize_entropy,
                    )
                else:
                    # TODO: If your efficient-kan build lacks regularization_loss(), keep kan_reg_strength=0.
                    pass

            loss.backward()
            optimizer.step()
            running_train_loss += loss.item() * xb.size(0)

        epoch_train_loss = running_train_loss / len(train_loader.dataset)

        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                preds = model(xb)
                val_loss = criterion(preds, yb)
                val_loss = val_loss + conv_l2_lambda * _conv_l2_penalty(model)
                running_val_loss += val_loss.item() * xb.size(0)

        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)

        if epoch_val_loss < best_val_loss - 1e-8:
            best_val_loss = epoch_val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, best_epoch


def predict_torch_regressor(model, data_loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in data_loader:
            xb = xb.to(DEVICE)
            batch_preds = model(xb)
            preds.append(batch_preds.cpu().numpy())
    return np.vstack(preds)

## Evaluation helpers

In [8]:
def evaluate_predictions(y_true, y_pred, target_names):
    results = {}
    for i, name in enumerate(target_names):
        mse = mean_squared_error(y_true[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
        r2 = r2_score(y_true[:, i], y_pred[:, i])
        results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    return results


def results_to_dataframe(results_dict):
    return pd.DataFrame(results_dict).T[['R2', 'RMSE', 'MAE']]


def long_metrics_dataframe(named_results):
    rows = []
    for model_name, results in named_results.items():
        for target, metrics in results.items():
            rows.append({
                'Model': model_name,
                'Target': target,
                'R2': metrics['R2'],
                'RMSE': metrics['RMSE'],
                'MAE': metrics['MAE'],
            })
    return pd.DataFrame(rows)


def build_r2_summary_table(named_results, target_names):
    rows = []
    for model_name, results in named_results.items():
        row = {'Model': model_name}
        for target in target_names:
            row[target] = results[target]['R2']
        row['Average R2'] = np.mean([results[t]['R2'] for t in target_names])
        rows.append(row)
    return pd.DataFrame(rows)


def print_results_block(model_name, results):
    print()
    print(f'{model_name} Results:')
    print(results_to_dataframe(results).to_string())

## Classical baselines and deep CV wrappers

In [9]:
def cross_validate_pls(X, y, splits, n_components=10):
    print(f"\n{'='*60}")
    print('PLS Regression')
    print(f"{'='*60}")
    y_pred_all = np.zeros_like(y, dtype=np.float32)

    for fold, (train_idx, val_idx) in enumerate(splits):
        print(f'\nFold {fold + 1}/{len(splits)}')
        X_train, X_val = X[train_idx], X[val_idx]
        y_train = y[train_idx]
        max_components = min(X_train.shape[0] - 1, X_train.shape[1], y_train.shape[1] if y_train.ndim > 1 else 1)
        n_comp_fold = max(1, min(n_components, max_components))
        model = PLSRegression(n_components=n_comp_fold)
        model.fit(X_train, y_train)
        y_pred_all[val_idx] = model.predict(X_val)
        print(f'  Using n_components={n_comp_fold}')

    return y_pred_all


def cross_validate_rf(X, y, splits, n_estimators=200, max_depth=None):
    print(f"\n{'='*60}")
    print('Random Forest')
    print(f"{'='*60}")
    y_pred_all = np.zeros_like(y, dtype=np.float32)

    for fold, (train_idx, val_idx) in enumerate(splits):
        print(f'\nFold {fold + 1}/{len(splits)}')
        X_train, X_val = X[train_idx], X[val_idx]
        y_train = y[train_idx]
        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=SEED,
            n_jobs=-1,
        )
        model.fit(X_train, y_train)
        y_pred_all[val_idx] = model.predict(X_val)

    return y_pred_all


def cross_validate_cnn_variant(
    X,
    y,
    splits,
    model_type='kan',
    epochs=100,
    batch_size=8,
    use_augmentation=False,
    dropout_rate=0.4,
    kan_hidden_dims=(64, 32),
    grid_size=5,
    spline_order=3,
    kan_reg_strength=0.0,
):
    print(f"\n{'='*60}")
    pretty_name = '1D-CNN + KAN' if model_type == 'kan' else '1D-CNN + MLP'
    print(f"{pretty_name}{' with Augmentation' if use_augmentation else ''}")
    print(f"{'='*60}")

    y_pred_all = np.zeros_like(y, dtype=np.float32)
    fold_histories = []
    fold_best_epochs = []
    fold_parameter_counts = []

    for fold, (train_idx, val_idx) in enumerate(splits):
        print(f'\nFold {fold + 1}/{len(splits)}')
        set_global_seed(SEED + fold)

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        if use_augmentation:
            X_train_aug, y_train_aug = augment_dataset(X_train, y_train, augmentation_factor=5)
            print(f'  Training samples: {len(X_train)} -> {len(X_train_aug)} (augmented)')
        else:
            X_train_aug, y_train_aug = X_train, y_train

        X_train_scaled, X_val_scaled, scaler = preprocess_spectra(X_train_aug, X_val)

        train_dataset = SpectraTensorDataset(X_train_scaled, y_train_aug)
        val_dataset = SpectraTensorDataset(X_val_scaled, y_val)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

        model = build_cnn_model(
            model_type=model_type,
            input_length=X_train_scaled.shape[1],
            n_outputs=y.shape[1],
            dropout_rate=dropout_rate,
            kan_hidden_dims=kan_hidden_dims,
            grid_size=grid_size,
            spline_order=spline_order,
        )

        model, history, best_epoch = train_torch_regressor(
            model,
            train_loader,
            val_loader,
            epochs=epochs,
            learning_rate=1e-3,
            patience=20,
            conv_l2_lambda=1e-3,
            kan_reg_strength=kan_reg_strength if model_type == 'kan' else 0.0,
        )

        y_pred_all[val_idx] = predict_torch_regressor(model, val_loader)
        fold_histories.append(history)
        fold_best_epochs.append(best_epoch + 1)
        fold_parameter_counts.append(count_parameters(model))
        print(f'  Best epoch: {best_epoch + 1}')

    return {
        'y_pred': y_pred_all,
        'histories': fold_histories,
        'best_epochs': fold_best_epochs,
        'parameter_counts': fold_parameter_counts,
        'model_type': model_type,
        'augmentation': use_augmentation,
    }

## Symbolic KAN helpers

In [10]:
def _safe_abs_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return 0.0
    corr = np.corrcoef(x, y)[0, 1]
    if np.isnan(corr):
        return 0.0
    return float(abs(corr))


def _make_symbolic_var_name(wavenumber, fallback_idx):
    if wavenumber is None:
        return f'x_{fallback_idx + 1}'
    wn = f'{float(wavenumber):.1f}'.replace('-', 'm').replace('.', 'p')
    return f'nu_{wn}'


def select_symbolic_features(X_train, y_train, top_k_features=4, wavenumbers=None):
    corrs = np.array([_safe_abs_corr(X_train[:, i], y_train) for i in range(X_train.shape[1])])
    top_k_features = min(top_k_features, X_train.shape[1])
    selected_idx = np.argsort(corrs)[::-1][:top_k_features]

    feature_report = []
    for rank, feat_idx in enumerate(selected_idx, start=1):
        wn = None if wavenumbers is None else float(wavenumbers[feat_idx])
        feature_report.append({
            'rank': rank,
            'feature_index': int(feat_idx),
            'wavenumber_cm^-1': wn,
            'abs_corr': float(corrs[feat_idx]),
            'var_name': _make_symbolic_var_name(wn, rank - 1),
        })
    return selected_idx, feature_report


def build_symbolic_dataset(X_train, y_train, X_val, y_val):
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    X_train_s = x_scaler.fit_transform(X_train)
    X_val_s = x_scaler.transform(X_val)
    y_train = np.asarray(y_train).reshape(-1, 1)
    y_val = np.asarray(y_val).reshape(-1, 1)
    y_train_s = y_scaler.fit_transform(y_train)
    y_val_s = y_scaler.transform(y_val)
    dataset = {
        'train_input': torch.tensor(X_train_s, dtype=torch.float32),
        'test_input': torch.tensor(X_val_s, dtype=torch.float32),
        'train_label': torch.tensor(y_train_s, dtype=torch.float32),
        'test_label': torch.tensor(y_val_s, dtype=torch.float32),
    }
    return dataset, x_scaler, y_scaler


def fit_small_pykan(
    dataset,
    n_inputs,
    hidden_width=3,
    grid_size=5,
    spline_order=3,
    steps=40,
    lamb=1e-2,
    symbolic_enabled=False,
    seed=SEED,
):
    model = SymbolicKAN(
        width=[n_inputs, hidden_width, 1],
        grid=grid_size,
        k=spline_order,
        seed=seed,
        device='cpu',
        auto_save=False,
        symbolic_enabled=symbolic_enabled,
    )

    if not symbolic_enabled and hasattr(model, 'speed'):
        model.speed()

    fit_result = model.fit(
        dataset,
        opt='LBFGS',
        steps=steps,
        lr=1.0,
        lamb=lamb if symbolic_enabled else 0.0,
        update_grid=True,
        grid_update_num=max(1, min(5, steps // 10 if steps >= 10 else 1)),
        start_grid_update_step=0,
        stop_grid_update_step=max(1, steps // 2),
        batch=-1,
        log=max(steps + 1, 999999),
    )
    return model, fit_result


def predict_small_pykan(model, X_scaled, y_scaler):
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    preds_scaled = model(X_tensor).detach().cpu().numpy().reshape(-1, 1)
    return y_scaler.inverse_transform(preds_scaled).reshape(-1)


def cross_validate_symbolic_kan(
    X,
    y,
    target_names,
    splits,
    wavenumbers=None,
    top_k_features=4,
    hidden_width=3,
    grid_size=5,
    spline_order=3,
    steps=40,
    lamb=1e-2,
):
    print()
    print(f"{'='*60}")
    print('Symbolic KAN (pykan on selected spectral features)')
    print(f"{'='*60}")

    y_pred_all = np.zeros_like(y, dtype=np.float32)
    feature_history = {name: [] for name in target_names}

    for fold, (train_idx, val_idx) in enumerate(splits):
        print()
        print(f'Fold {fold + 1}/{len(splits)}')
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        for target_idx, target_name in enumerate(target_names):
            selected_idx, feature_report = select_symbolic_features(
                X_train,
                y_train[:, target_idx],
                top_k_features=top_k_features,
                wavenumbers=wavenumbers,
            )
            feature_history[target_name].append(feature_report)
            X_train_sel = X_train[:, selected_idx]
            X_val_sel = X_val[:, selected_idx]
            dataset, x_scaler, y_scaler = build_symbolic_dataset(
                X_train_sel,
                y_train[:, target_idx],
                X_val_sel,
                y_val[:, target_idx],
            )
            model, _ = fit_small_pykan(
                dataset,
                n_inputs=X_train_sel.shape[1],
                hidden_width=hidden_width,
                grid_size=grid_size,
                spline_order=spline_order,
                steps=steps,
                lamb=lamb,
                symbolic_enabled=False,
                seed=SEED + fold * 100 + target_idx,
            )
            X_val_scaled = x_scaler.transform(X_val_sel)
            preds = predict_small_pykan(model, X_val_scaled, y_scaler)
            y_pred_all[val_idx, target_idx] = preds.astype(np.float32)
            wn_summary = []
            for row in feature_report:
                if row['wavenumber_cm^-1'] is not None:
                    wn_summary.append(f"{row['wavenumber_cm^-1']:.1f}")
                else:
                    wn_summary.append(str(row['feature_index']))
            print(f"  {target_name}: top features -> {', '.join(wn_summary)}")

    return y_pred_all, feature_history


def _feature_report_to_text(feature_report):
    chunks = []
    for row in feature_report:
        if row['wavenumber_cm^-1'] is None:
            chunks.append(f"{row['var_name']} (idx={row['feature_index']}, |corr|={row['abs_corr']:.3f})")
        else:
            chunks.append(
                f"{row['var_name']} ({row['wavenumber_cm^-1']:.1f} cm^-1, |corr|={row['abs_corr']:.3f})"
            )
    return '; '.join(chunks)


def discover_symbolic_equations(
    X,
    y,
    target_names,
    wavenumbers=None,
    top_k_features=4,
    hidden_width=3,
    grid_size=5,
    spline_order=3,
    steps=60,
    fine_tune_steps=20,
    lamb=1e-2,
    node_th=1e-2,
    edge_th=3e-2,
    weight_simple=0.85,
    symbolic_lib=None,
):
    if symbolic_lib is None:
        symbolic_lib = ['x', 'x^2', 'x^3', 'abs', 'tanh', 'gaussian', 'exp']

    reports = []
    for target_idx, target_name in enumerate(target_names):
        selected_idx, feature_report = select_symbolic_features(
            X,
            y[:, target_idx],
            top_k_features=top_k_features,
            wavenumbers=wavenumbers,
        )
        X_sel = X[:, selected_idx]
        dataset, x_scaler, y_scaler = build_symbolic_dataset(X_sel, y[:, target_idx], X_sel, y[:, target_idx])
        equation_str = 'Equation extraction failed.'
        equation_latex = 'N/A'
        fit_r2 = np.nan
        extraction_status = 'ok'

        try:
            model, _ = fit_small_pykan(
                dataset,
                n_inputs=X_sel.shape[1],
                hidden_width=hidden_width,
                grid_size=grid_size,
                spline_order=spline_order,
                steps=steps,
                lamb=lamb,
                symbolic_enabled=True,
                seed=SEED + 1000 + target_idx,
            )
            try:
                model = model.prune(node_th=node_th, edge_th=edge_th)
            except Exception as prune_exc:
                extraction_status = f'prune_warning: {prune_exc}'
            try:
                model.fit(
                    dataset,
                    opt='LBFGS',
                    steps=fine_tune_steps,
                    lr=1.0,
                    lamb=lamb,
                    update_grid=False,
                    batch=-1,
                    log=max(fine_tune_steps + 1, 999999),
                )
            except Exception as fine_tune_exc:
                extraction_status = f'{extraction_status}; fine_tune_warning: {fine_tune_exc}'

            model.auto_symbolic(lib=symbolic_lib, verbose=1, weight_simple=weight_simple, r2_threshold=0.0)
            var_names = [row['var_name'] for row in feature_report]
            formulas, variables = model.symbolic_formula(
                var=var_names,
                normalizer=[x_scaler.mean_, x_scaler.scale_],
                output_normalizer=[y_scaler.mean_, y_scaler.scale_],
            )
            expr = formulas[0]
            equation_str = str(expr)
            equation_latex = sp.latex(expr)
            preds_scaled = model(dataset['train_input']).detach().cpu().numpy().reshape(-1, 1)
            preds = y_scaler.inverse_transform(preds_scaled).reshape(-1)
            fit_r2 = r2_score(y[:, target_idx], preds)
        except Exception as exc:
            extraction_status = f'error: {exc}'

        reports.append({
            'target': target_name,
            'feature_report': feature_report,
            'feature_summary': _feature_report_to_text(feature_report),
            'equation_str': equation_str,
            'equation_latex': equation_latex,
            'full_data_r2': fit_r2,
            'status': extraction_status,
        })

    return reports


def summarize_symbolic_equations(reports):
    rows = []
    print()
    print(f"{'='*60}")
    print('SYMBOLIC EQUATION SUMMARY')
    print(f"{'='*60}")
    for report in reports:
        print()
        print(f"Target: {report['target']}")
        print(f"Selected features: {report['feature_summary']}")
        print(f"Equation: {report['equation_str']}")
        print(f"Full-data symbolic fit R²: {report['full_data_r2']}")
        print(f"Status: {report['status']}")
        rows.append({
            'Target': report['target'],
            'Selected Features': report['feature_summary'],
            'Equation': report['equation_str'],
            'Equation LaTeX': report['equation_latex'],
            'Full-data R2': report['full_data_r2'],
            'Status': report['status'],
        })
    return pd.DataFrame(rows)

## Presentation plots and table exports

In [11]:
def plot_ftir_overview(X, wavenumbers, y, output_dir, title='FTIR Spectrum Overview'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    for i in range(X.shape[0]):
        ax.plot(wavenumbers, X[i], alpha=0.25)
    ax.set_title('All spectra')
    ax.set_xlabel('Wavenumber (cm$^{-1}$)')
    ax.set_ylabel('Absorbance (a.u.)')
    ax.invert_xaxis()
    ax.grid(True, alpha=0.25)

    ax = axes[1]
    sort_idx = np.argsort(y[:, 0])
    selected = np.unique(np.linspace(0, len(sort_idx) - 1, min(6, len(sort_idx))).astype(int))
    for idx in selected:
        sample_idx = sort_idx[idx]
        label = f'{y[sample_idx,0]:.2f} M'
        ax.plot(wavenumbers, X[sample_idx], label=label)
    ax.set_title('Representative spectra by LiPF6 molarity')
    ax.set_xlabel('Wavenumber (cm$^{-1}$)')
    ax.set_ylabel('Absorbance (a.u.)')
    ax.invert_xaxis()
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    out_path = Path(output_dir) / 'slide02_ftir_overview.png'
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return out_path


def plot_predicted_vs_actual_grid(y_true, y_pred, target_names, output_dir, file_name, title):
    n_targets = y_true.shape[1]
    ncols = 2
    nrows = math.ceil(n_targets / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 5 * nrows))
    axes = np.array(axes).reshape(-1)

    rows = []
    for i, name in enumerate(target_names):
        ax = axes[i]
        ax.scatter(y_true[:, i], y_pred[:, i], alpha=0.75, s=55)
        min_val = min(y_true[:, i].min(), y_pred[:, i].min())
        max_val = max(y_true[:, i].max(), y_pred[:, i].max())
        ax.plot([min_val, max_val], [min_val, max_val], '--', lw=1.8)
        r2 = r2_score(y_true[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
        mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rows.append({'Target': name, 'R2': r2, 'RMSE': rmse, 'MAE': mae})
        ax.set_title(f'{name}\nR² = {r2:.3f}, RMSE = {rmse:.4f}')
        ax.set_xlabel(f'Actual {name}')
        ax.set_ylabel(f'Predicted {name}')
        ax.grid(True, alpha=0.25)

    for j in range(n_targets, len(axes)):
        axes[j].axis('off')

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    out_path = Path(output_dir) / file_name
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return pd.DataFrame(rows), out_path


def plot_training_curves(histories, output_dir, file_name, title):
    n_folds = len(histories)
    ncols = 2
    nrows = math.ceil(n_folds / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4.5 * nrows))
    axes = np.array(axes).reshape(-1)

    for fold_idx, history in enumerate(histories):
        ax = axes[fold_idx]
        epochs = np.arange(1, len(history['train_loss']) + 1)
        ax.plot(epochs, history['train_loss'], label='Train')
        ax.plot(epochs, history['val_loss'], '--', label='Validation')
        best_epoch = int(np.argmin(history['val_loss'])) + 1
        ax.axvline(best_epoch, linestyle=':', linewidth=1.5)
        ax.set_title(f'Fold {fold_idx + 1} (best epoch = {best_epoch})')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss (MSE + regularization)')
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)

    for j in range(n_folds, len(axes)):
        axes[j].axis('off')

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    out_path = Path(output_dir) / file_name
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return out_path


def plot_r2_ablation(cnn_mlp_results, cnn_kan_results, target_names, output_dir, file_name):
    mlp_r2 = [cnn_mlp_results[t]['R2'] for t in target_names]
    kan_r2 = [cnn_kan_results[t]['R2'] for t in target_names]
    x = np.arange(len(target_names))
    width = 0.36

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width / 2, mlp_r2, width, label='CNN+MLP')
    ax.bar(x + width / 2, kan_r2, width, label='CNN+KAN')
    ax.set_xticks(x)
    ax.set_xticklabels(target_names, rotation=15, ha='right')
    ax.set_ylabel('R²')
    ax.set_title('CNN+MLP vs CNN+KAN — Per-target R² Ablation')
    ax.grid(True, axis='y', alpha=0.25)
    ax.legend()
    plt.tight_layout()

    out_path = Path(output_dir) / file_name
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

    df = pd.DataFrame({
        'Target': target_names,
        'CNN+MLP R2': mlp_r2,
        'CNN+KAN R2': kan_r2,
        'Delta (KAN-MLP)': np.array(kan_r2) - np.array(mlp_r2),
    })
    return df, out_path


def render_summary_table(df, output_path, title, best_highlight='max'):
    df_render = df.copy()
    numeric_cols = [c for c in df_render.columns if c != 'Model']
    fig_h = max(2.8, 0.5 * len(df_render) + 1.8)
    fig_w = max(8.0, 1.8 * len(df_render.columns) + 1.0)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')

    cell_text = []
    for _, row in df_render.iterrows():
        vals = []
        for col in df_render.columns:
            val = row[col]
            if isinstance(val, (float, np.floating)):
                vals.append(f'{val:.3f}')
            else:
                vals.append(str(val))
        cell_text.append(vals)

    table = ax.table(
        cellText=cell_text,
        colLabels=df_render.columns.tolist(),
        loc='center',
        cellLoc='center',
        colLoc='center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.5)

    for col_idx, col_name in enumerate(df_render.columns):
        table[(0, col_idx)].set_text_props(weight='bold')

    for col_name in numeric_cols:
        col_idx = df_render.columns.get_loc(col_name)
        values = df_render[col_name].astype(float).values
        best_val = values.max() if best_highlight == 'max' else values.min()
        for row_idx, val in enumerate(values, start=1):
            if np.isclose(val, best_val):
                table[(row_idx, col_idx)].set_facecolor('#d9ead3')

    ax.set_title(title, pad=20)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return output_path


def plot_symbolic_equations_panel(symbolic_df, output_dir, file_name):
    fig, axes = plt.subplots(len(symbolic_df), 1, figsize=(14, max(3.0, 2.6 * len(symbolic_df))))
    if len(symbolic_df) == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, symbolic_df.iterrows()):
        ax.axis('off')
        text = (
            f"Target: {row['Target']}\n"
            f"Selected features: {row['Selected Features']}\n"
            f"Equation: {row['Equation']}\n"
            f"Full-data symbolic R²: {row['Full-data R2']:.3f} | Status: {row['Status']}"
        )
        wrapped = '\n'.join(textwrap.wrap(text, width=110, break_long_words=False))
        ax.text(0.01, 0.95, wrapped, va='top', ha='left', fontsize=10, family='monospace')

    plt.suptitle('Symbolic KAN Equations', fontsize=14)
    plt.tight_layout()
    out_path = Path(output_dir) / file_name
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return out_path


def save_powerpoint_tables(output_dir, named_results, target_names, primary_scatter_df, symbolic_df, ablation_df, comparison_notes):
    output_dir = Path(output_dir)
    full_metrics_df = long_metrics_dataframe(named_results)
    r2_summary_df = build_r2_summary_table(named_results, target_names)

    full_metrics_path = output_dir / 'ppt_table_full_metrics_long.csv'
    r2_summary_path = output_dir / 'ppt_table_model_r2_summary.csv'
    scatter_metrics_path = output_dir / 'ppt_table_cnn_kan_scatter_metrics.csv'
    ablation_path = output_dir / 'ppt_table_ablation_r2.csv'
    symbolic_path = output_dir / 'ppt_table_symbolic_equations.csv'
    notes_path = output_dir / 'ppt_table_notes.txt'

    full_metrics_df.to_csv(full_metrics_path, index=False)
    r2_summary_df.to_csv(r2_summary_path, index=False)
    primary_scatter_df.to_csv(scatter_metrics_path, index=False)
    ablation_df.to_csv(ablation_path, index=False)
    symbolic_df.to_csv(symbolic_path, index=False)

    with open(notes_path, 'w') as f:
        f.write(comparison_notes)

    return {
        'full_metrics_csv': full_metrics_path,
        'r2_summary_csv': r2_summary_path,
        'scatter_metrics_csv': scatter_metrics_path,
        'ablation_csv': ablation_path,
        'symbolic_csv': symbolic_path,
        'notes_txt': notes_path,
    }

## Main pipeline

In [12]:
def run_full_pipeline(
    X,
    y,
    target_names,
    wavenumbers=None,
    output_dir='presentation_artifacts',
    kan_hidden_dims=(64, 32),
    grid_size=5,
    spline_order=3,
    kan_reg_strength=0.0,
    symbolic_top_k_features=4,
    symbolic_hidden_width=3,
    symbolic_steps=40,
    symbolic_final_steps=60,
    symbolic_lamb=1e-2,
    n_folds=5,
):
    output_dir = ensure_dir(output_dir)
    assert y.shape[1] == len(target_names), (
        f'Shape mismatch: y has {y.shape[1]} columns but target_names has {len(target_names)} entries.'
    )

    print()
    print(f"{'#'*60}")
    print('FTIR MULTI-COMPONENT PREDICTION PIPELINE')
    print(f"{'#'*60}")
    print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
    print(f'Targets: {target_names}')

    ftir_overview_path = None
    if wavenumbers is not None:
        ftir_overview_path = plot_ftir_overview(X, wavenumbers, y, output_dir)

    splits = make_cv_splits(X, n_folds=n_folds, seed=SEED)
    X_scaled, _ = preprocess_spectra(X)

    y_pred_pls = cross_validate_pls(X_scaled, y, splits=splits, n_components=10)
    results_pls = evaluate_predictions(y, y_pred_pls, target_names)
    print_results_block('PLS', results_pls)

    y_pred_rf = cross_validate_rf(X_scaled, y, splits=splits, n_estimators=200, max_depth=None)
    results_rf = evaluate_predictions(y, y_pred_rf, target_names)
    print_results_block('Random Forest', results_rf)

    cnn_mlp_no_aug = cross_validate_cnn_variant(
        X, y, splits=splits, model_type='mlp', epochs=100, batch_size=8, use_augmentation=False
    )
    results_cnn_mlp = evaluate_predictions(y, cnn_mlp_no_aug['y_pred'], target_names)
    print_results_block('1D-CNN + MLP (No Augmentation)', results_cnn_mlp)

    cnn_mlp_aug = cross_validate_cnn_variant(
        X, y, splits=splits, model_type='mlp', epochs=100, batch_size=8, use_augmentation=True
    )
    results_cnn_mlp_aug = evaluate_predictions(y, cnn_mlp_aug['y_pred'], target_names)
    print_results_block('1D-CNN + MLP (With Augmentation)', results_cnn_mlp_aug)

    cnn_kan_no_aug = cross_validate_cnn_variant(
        X,
        y,
        splits=splits,
        model_type='kan',
        epochs=100,
        batch_size=8,
        use_augmentation=False,
        kan_hidden_dims=kan_hidden_dims,
        grid_size=grid_size,
        spline_order=spline_order,
        kan_reg_strength=kan_reg_strength,
    )
    results_cnn_kan = evaluate_predictions(y, cnn_kan_no_aug['y_pred'], target_names)
    print_results_block('1D-CNN + KAN (No Augmentation)', results_cnn_kan)

    cnn_kan_aug = cross_validate_cnn_variant(
        X,
        y,
        splits=splits,
        model_type='kan',
        epochs=100,
        batch_size=8,
        use_augmentation=True,
        kan_hidden_dims=kan_hidden_dims,
        grid_size=grid_size,
        spline_order=spline_order,
        kan_reg_strength=kan_reg_strength,
    )
    results_cnn_kan_aug = evaluate_predictions(y, cnn_kan_aug['y_pred'], target_names)
    print_results_block('1D-CNN + KAN (With Augmentation)', results_cnn_kan_aug)

    y_pred_symbolic_kan, symbolic_feature_history = cross_validate_symbolic_kan(
        X,
        y,
        target_names,
        splits=splits,
        wavenumbers=wavenumbers,
        top_k_features=symbolic_top_k_features,
        hidden_width=symbolic_hidden_width,
        grid_size=grid_size,
        spline_order=spline_order,
        steps=symbolic_steps,
        lamb=symbolic_lamb,
    )
    results_symbolic_kan = evaluate_predictions(y, y_pred_symbolic_kan, target_names)
    print_results_block('Symbolic KAN', results_symbolic_kan)

    symbolic_equations = discover_symbolic_equations(
        X,
        y,
        target_names,
        wavenumbers=wavenumbers,
        top_k_features=symbolic_top_k_features,
        hidden_width=symbolic_hidden_width,
        grid_size=grid_size,
        spline_order=spline_order,
        steps=symbolic_final_steps,
        fine_tune_steps=max(10, symbolic_final_steps // 3),
        lamb=symbolic_lamb,
    )
    symbolic_equation_summary = summarize_symbolic_equations(symbolic_equations)
    display(symbolic_equation_summary)

    named_results = {
        'PLS': results_pls,
        'Random Forest': results_rf,
        'CNN+MLP (No Aug)': results_cnn_mlp,
        'CNN+MLP (Aug)': results_cnn_mlp_aug,
        'CNN+KAN (No Aug)': results_cnn_kan,
        'CNN+KAN (Aug)': results_cnn_kan_aug,
        'Symbolic KAN': results_symbolic_kan,
    }

    r2_summary_df = build_r2_summary_table(named_results, target_names)
    comparison = {row['Model']: row['Average R2'] for _, row in r2_summary_df.iterrows()}

    print()
    print(f"{'='*60}")
    print('MODEL COMPARISON (Average R² Score)')
    print(f"{'='*60}")
    for model_name, avg_r2 in comparison.items():
        print(f'{model_name:22s}: {avg_r2:.3f}')
    best_model_name = max(comparison, key=comparison.get)
    print()
    print(f'Best model: {best_model_name}')

    primary_kan_name = 'CNN+KAN (No Aug)'
    primary_kan_pred = cnn_kan_no_aug['y_pred']
    primary_kan_results = results_cnn_kan
    primary_kan_histories = cnn_kan_no_aug['histories']
    if comparison['CNN+KAN (Aug)'] > comparison['CNN+KAN (No Aug)']:
        primary_kan_name = 'CNN+KAN (Aug)'
        primary_kan_pred = cnn_kan_aug['y_pred']
        primary_kan_results = results_cnn_kan_aug
        primary_kan_histories = cnn_kan_aug['histories']

    slide9_metrics_df, pred_actual_path = plot_predicted_vs_actual_grid(
        y,
        primary_kan_pred,
        target_names,
        output_dir,
        'slide09_predicted_vs_actual_primary_kan.png',
        f'Predicted vs Actual — {primary_kan_name}',
    )

    training_curves_path = plot_training_curves(
        primary_kan_histories,
        output_dir,
        'slide11_training_curves_primary_kan.png',
        f'Training Curves — {primary_kan_name}',
    )

    ablation_df, ablation_plot_path = plot_r2_ablation(
        results_cnn_mlp,
        results_cnn_kan,
        target_names,
        output_dir,
        'slide11_ablation_cnn_mlp_vs_cnn_kan.png',
    )

    symbolic_panel_path = plot_symbolic_equations_panel(
        symbolic_equation_summary,
        output_dir,
        'slide10_symbolic_equations_panel.png',
    )

    summary_table_img_path = render_summary_table(
        r2_summary_df,
        Path(output_dir) / 'slide08_model_r2_summary_table.png',
        'Performance Comparison (5-Fold CV) — R² Summary',
        best_highlight='max',
    )

    primary_kan_metrics_df = results_to_dataframe(primary_kan_results).reset_index().rename(columns={'index': 'Target'})
    primary_kan_metrics_df.to_csv(Path(output_dir) / 'ppt_table_primary_kan_metrics_full.csv', index=False)

    comparison_notes = (
        f'Best overall model by average R²: {best_model_name}\n'
        f'Primary KAN figure used for Slide 9: {primary_kan_name}\n'
        f'CNN+MLP mean parameter count: {np.mean(cnn_mlp_no_aug["parameter_counts"]):.0f}\n'
        f'CNN+KAN mean parameter count: {np.mean(cnn_kan_no_aug["parameter_counts"]):.0f}\n'
        f'Average best epoch for primary KAN: {np.mean([np.argmin(h["val_loss"]) + 1 for h in primary_kan_histories]):.1f}\n'
    )

    exported_tables = save_powerpoint_tables(
        output_dir,
        named_results,
        target_names,
        slide9_metrics_df,
        symbolic_equation_summary,
        ablation_df,
        comparison_notes,
    )

    export_manifest = pd.DataFrame([
        {'Artifact': 'FTIR overview plot', 'Path': str(ftir_overview_path) if ftir_overview_path else 'N/A'},
        {'Artifact': 'Slide 8 R² summary table image', 'Path': str(summary_table_img_path)},
        {'Artifact': 'Slide 9 predicted-vs-actual plot', 'Path': str(pred_actual_path)},
        {'Artifact': 'Slide 10 symbolic equations panel', 'Path': str(symbolic_panel_path)},
        {'Artifact': 'Slide 11 training curves', 'Path': str(training_curves_path)},
        {'Artifact': 'Slide 11 ablation plot', 'Path': str(ablation_plot_path)},
        {'Artifact': 'PowerPoint tables bundle', 'Path': str(exported_tables['r2_summary_csv'])},
    ])
    display(export_manifest)

    return {
        'named_results': named_results,
        'comparison': comparison,
        'best_model': best_model_name,
        'primary_kan_name': primary_kan_name,
        'primary_kan_metrics_df': primary_kan_metrics_df,
        'slide9_metrics_df': slide9_metrics_df,
        'symbolic_feature_history': symbolic_feature_history,
        'symbolic_equations': symbolic_equations,
        'symbolic_equation_summary': symbolic_equation_summary,
        'r2_summary_df': r2_summary_df,
        'ablation_df': ablation_df,
        'exported_tables': exported_tables,
        'export_manifest': export_manifest,
        'artifact_paths': {
            'ftir_overview': ftir_overview_path,
            'slide08_table_png': summary_table_img_path,
            'slide09_pred_vs_actual_png': pred_actual_path,
            'slide10_symbolic_png': symbolic_panel_path,
            'slide11_training_png': training_curves_path,
            'slide11_ablation_png': ablation_plot_path,
            'primary_kan_metrics_csv': Path(output_dir) / 'ppt_table_primary_kan_metrics_full.csv',
        },
    }

## Configuration and execution

In [13]:
# Replace this with the correct Colab/Drive path for your spectra directory.
base_data_path = '/content/drive/MyDrive/Machine Learning/Spectra'

# Example alternative for local testing:
# base_data_path = './Spectra'

ftir_spectra, ftir_labels = load_ftir_data(base_data_path)
print(f'\nTotal spectra loaded: {len(ftir_spectra)}')
print(f'Total labels loaded: {len(ftir_labels)}')

if ftir_spectra and ftir_labels:
    X, y, wavenumbers = build_clean_dataset(ftir_spectra, ftir_labels)
    y = y[:, :4]  # [LiPF6 Molarity, LiPF6 mole %, EC mole %, EMC mole %]
    target_names = ['LiPF6 Molarity (mol/L)', 'Mole % LiPF6', 'Mole % EC', 'Mole % EMC']

    print(f"{'='*60}")
    print('CLEANED DATASET:')
    print(f"{'='*60}")
    print(f'X shape: {X.shape}, dtype: {X.dtype}')
    print(f'y shape: {y.shape}, dtype: {y.dtype}')
    print(f'Wavenumber axis shape: {wavenumbers.shape}')
    print(f'Wavenumber range: {wavenumbers.min():.1f} to {wavenumbers.max():.1f} cm^-1')

    results = run_full_pipeline(
        X,
        y,
        target_names,
        wavenumbers=wavenumbers,
        output_dir='presentation_artifacts',
        kan_hidden_dims=(64, 32),
        grid_size=5,
        spline_order=3,
        kan_reg_strength=0.0,
        symbolic_top_k_features=4,
        symbolic_hidden_width=3,
        symbolic_steps=40,
        symbolic_final_steps=60,
        symbolic_lamb=1e-2,
        n_folds=5,
    )

    print('\nPowerPoint-ready artifacts were written to: presentation_artifacts/')
else:
    print('\nNo data loaded. Fix base_data_path before running the pipeline.')

Successfully processed 6 sets of FTIR data.

Total spectra loaded: 6
Total labels loaded: 6
CLEANED DATASET:
X shape: (42, 3312), dtype: float64
y shape: (42, 4), dtype: float64
Wavenumber axis shape: (3312,)
Wavenumber range: 400.2 to 1996.5 cm^-1

############################################################
FTIR MULTI-COMPONENT PREDICTION PIPELINE
############################################################
Dataset: 42 samples, 3312 features
Targets: ['LiPF6 Molarity (mol/L)', 'Mole % LiPF6', 'Mole % EC', 'Mole % EMC']

PLS Regression

Fold 1/5
  Using n_components=4

Fold 2/5
  Using n_components=4

Fold 3/5
  Using n_components=4

Fold 4/5
  Using n_components=4

Fold 5/5
  Using n_components=4

PLS Results:
                              R2      RMSE       MAE
LiPF6 Molarity (mol/L)  0.946660  0.124504  0.094384
Mole % LiPF6            0.905726  0.012753  0.009789
Mole % EC               0.193303  0.242076  0.203849
Mole % EMC              0.197218  0.234350  0.196638

Random Fores

| train_loss: 1.42e-01 | test_loss: 1.99e-01 | reg: 0.00e+00 | : 100%|█| 40/40 [00:04<00:00,  9.23it


  LiPF6 Molarity (mol/L): top features -> 1186.5, 1186.0, 1187.0, 779.6


| train_loss: 2.11e-01 | test_loss: 1.84e-01 | reg: 0.00e+00 | : 100%|█| 40/40 [00:02<00:00, 14.60it


  Mole % LiPF6: top features -> 745.4, 744.9, 745.8, 744.4


...

fixing (0,0,0) with 0
fixing (0,0,1) with x, r2=0.04742615297436714, c=1
fixing (0,0,2) with x, r2=0.010105593129992485, c=1
fixing (0,1,0) with 0
fixing (0,1,1) with x, r2=0.0002427808358334005, c=1
fixing (0,1,2) with x, r2=0.0003114222490694374, c=1
fixing (0,2,0) with 0
fixing (0,2,1) with x, r2=0.6124674081802368, c=1
fixing (0,2,2) with x, r2=0.20881816744804382, c=1
fixing (0,3,0) with 0
fixing (0,3,1) with x, r2=0.39648720622062683, c=1
fixing (0,3,2) with x, r2=0.7796129584312439, c=1
fixing (1,0,0) with 0
fixing (1,1,0) with x, r2=0.05097723752260208, c=1
fixing (1,2,0) with x, r2=0.36556175351142883, c=1

SYMBOLIC EQUATION SUMMARY

Target: LiPF6 Molarity (mol/L)
Selected features: nu_779p6 (779.6 cm^-1, |corr|=0.986); nu_780p1 (780.1 cm^-1, |corr|=0.985); nu_779p1 (779.1 cm^-1, |corr|=0.984); nu_740p1 (740.1 cm^-1, |corr|=0.983)
Equation: 12.2559860661487*nu_740p1 + 0.00199761353995233*nu_779p1 - 0.00230987401686014*nu_779p6 + 0.00742714766779138*nu_780p1 + 0.197025297877003

,Target,Selected Features,Equation,Equation LaTeX,Full-data R2,Status
0,LiPF6 Molarity (mol/L),"nu_779p6 (779.6 cm^-1, |corr|=0.986); nu_780p1...",12.2559860661487*nu_740p1 + 0.0019976135399523...,12.2559860661487 \nu_{740p1} + 0.0019976135399...,0.964040,prune_warning: [Errno 2] No such file or direc...
1,Mole % LiPF6,"nu_745p4 (745.4 cm^-1, |corr|=0.965); nu_744p9...",0.0407699698994525*nu_744p4 + 1.11662807770996...,0.0407699698994525 \nu_{744p4} + 1.11662807770...,0.931826,prune_warning: [Errno 2] No such file or direc...
2,Mole % EC,"nu_1733p2 (1733.2 cm^-1, |corr|=0.606); nu_173...",2.97050569385062*nu_1732p8 - 0.004206402132236...,2.97050569385062 \nu_{1732p8} - 0.004206402132...,0.198546,prune_warning: [Errno 2] No such file or direc...
3,Mole % EMC,"nu_1733p2 (1733.2 cm^-1, |corr|=0.617); nu_173...",-1.08315387437753*nu_1732p3 - 0.00011427966291...,- 1.08315387437753 \nu_{1732p3} - 0.0001142796...,0.230641,prune_warning: [Errno 2] No such file or direc...



MODEL COMPARISON (Average R² Score)
PLS                   : 0.561
Random Forest         : 0.451
CNN+MLP (No Aug)      : 0.527
CNN+MLP (Aug)         : 0.443
CNN+KAN (No Aug)      : 0.636
CNN+KAN (Aug)         : 0.505
Symbolic KAN          : -5.833

Best model: CNN+KAN (No Aug)


,Artifact,Path
0,FTIR overview plot,presentation_artifacts/slide02_ftir_overview.png
1,Slide 8 R² summary table image,presentation_artifacts/slide08_model_r2_summar...
2,Slide 9 predicted-vs-actual plot,presentation_artifacts/slide09_predicted_vs_ac...
3,Slide 10 symbolic equations panel,presentation_artifacts/slide10_symbolic_equati...
4,Slide 11 training curves,presentation_artifacts/slide11_training_curves...
5,Slide 11 ablation plot,presentation_artifacts/slide11_ablation_cnn_ml...
6,PowerPoint tables bundle,presentation_artifacts/ppt_table_model_r2_summ...



PowerPoint-ready artifacts were written to: presentation_artifacts/
